In [ ]:
from datasets import load_dataset

dataset = load_dataset("tweet_eval", "sentiment")

In [ ]:
dataset = dataset.shuffle(seed=42)

small_dataset = {
    "train": dataset["train"].select(range(10000)),
    "validation": dataset["validation"].select(range(2000)),
    "test": dataset["test"].select(range(2000)),
}

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = {
    split: small_dataset[split].map(tokenize, batched=True)
    for split in small_dataset
}

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"],
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate(tokenized_dataset["test"])

In [ ]:
def run_experiment(lr, batch_size, epochs):
    args = TrainingArguments(
        output_dir=f"./results_lr_{lr}",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        evaluation_strategy="epoch",
        logging_dir="./logs",
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base",
        num_labels=3
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    return trainer.evaluate()

In [ ]:
results = []

results.append(run_experiment(2e-5, 16, 2))
results.append(run_experiment(1e-5, 16, 3))
results.append(run_experiment(3e-5, 32, 2))

print(results)